# KV Cache Migration vs Re-computing — 정량 분석

figure 가 아니라 **수치 분석**용 노트북. 두 방식의 시간 차이를 context length / model / instance 별로 비교한다.

**데이터** (`figures/plot_comparison.py` 와 동일한 매칭 규칙)
- `kv_cache_migration.csv` : `Context Length, Model Size, Source/Destination Instance, KV Cache Size (MiB), KV Cache Transfer Time (ms), Network Bandwidth (GB/s)`
- `recomputing.csv` : `Context Length, Model Size, Instance Type, Re-Computing Latency (ms)`

**비교의 의미 (migration 의사결정)**
- 요청을 instance A→B 로 옮길 때: ① B 에서 KV 재계산(recompute) vs ② A 에서 B 로 KV 전송(migration).
- 따라서 **recompute 는 항상 destination instance 기준** 으로 매칭한다.
- instance 매핑: `L4 (g6.xlarge)` ↔ `g6.xlarge`, `L40S (g6e.xlarge)` ↔ `g6e.xlarge`.
- **70B 매핑**: KV migration 의 `70B` 는 *해당 instance 에 올라가는 layer 수만큼의 KV* 다 (g6.xlarge=2 layer, g6e.xlarge=8 layer). recompute 의 `70B(2layer)`/`70B(8layer)` 와 layer 수가 이미 일치하므로 **정규화 없이 직접 비교**한다.
- recompute 가 `NaN` = 해당 instance 에서 그 context 가 OOM/측정불가 → **migration 이 사실상 유일한 선택**.

**부호 규약**: `diff_ms = migrate_ms - recompute_ms` (양수면 migration 이 더 느림), `ratio = migrate_ms / recompute_ms` (>1 이면 recompute 가 빠름).

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

kv = pd.read_csv('kv_cache_migration.csv')
rc = pd.read_csv('recomputing.csv')

# --- recompute 쪽 정리: instance 약칭 + 70B layer 라벨 통합 ---
INST = {'L4 (g6.xlarge)': 'g6.xlarge', 'L40S (g6e.xlarge)': 'g6e.xlarge'}
rc['dst'] = rc['Instance Type'].map(INST)
rc['model'] = rc['Model Size'].apply(lambda m: '70B' if m.startswith('70B') else m)
rc2 = rc.rename(columns={'Context Length': 'ctx',
                         'Re-Computing Latency (ms)': 'recompute_ms'})[['model', 'dst', 'ctx', 'recompute_ms']]

# --- migration 쪽 정리 ---
kv2 = kv.rename(columns={'Model Size': 'model', 'Source Instance': 'src',
                         'Destination Instance': 'dst', 'Context Length': 'ctx',
                         'KV Cache Transfer Time (ms)': 'migrate_ms',
                         'KV Cache Size (MiB)': 'kv_MiB',
                         'Network Bandwidth (GB/s)': 'bw'})

# --- recompute(=destination 기준) 와 merge ---
df = kv2.merge(rc2, on=['model', 'dst', 'ctx'], how='left')
df['pair'] = df['src'] + '->' + df['dst']

# --- 핵심 지표 ---
df['diff_ms'] = df['migrate_ms'] - df['recompute_ms']
df['ratio']   = df['migrate_ms'] / df['recompute_ms']
df['faster']  = np.where(df['recompute_ms'].isna(), 'migration (recompute OOM)',
                np.where(df['migrate_ms'] < df['recompute_ms'], 'migration', 'recompute'))
df['kv_GB']   = df['kv_MiB'] / 1024
# migration 이 recompute 와 같아지려면 필요한 대역폭 (= bw_actual * ratio)
df['breakeven_bw_GBps'] = df['kv_GB'] / (df['recompute_ms'] / 1000)

df = df.sort_values(['model', 'pair', 'ctx']).reset_index(drop=True)
print(f'merged rows: {len(df)}  (recompute 있음 {df.recompute_ms.notna().sum()}, OOM {df.recompute_ms.isna().sum()})')
print('models:', sorted(df.model.unique()), '| pairs:', sorted(df.pair.unique()))
df.head()

merged rows: 100  (recompute 있음 95, OOM 5)
models: ['3B', '70B', '8B'] | pairs: ['g6.xlarge->g6.xlarge', 'g6.xlarge->g6e.xlarge', 'g6e.xlarge->g6.xlarge', 'g6e.xlarge->g6e.xlarge']


,ctx,model,src,dst,kv_MiB,migrate_ms,bw,recompute_ms,pair,diff_ms,ratio,faster,kv_GB,breakeven_bw_GBps
0,128,3B,g6.xlarge,g6.xlarge,14,44.98,0.30,37.80,g6.xlarge->g6.xlarge,7.18,1.19,recompute,0.01,0.36
1,256,3B,g6.xlarge,g6.xlarge,28,94.23,0.29,38.00,g6.xlarge->g6.xlarge,56.23,2.48,recompute,0.03,0.72
2,512,3B,g6.xlarge,g6.xlarge,56,188.91,0.29,45.90,g6.xlarge->g6.xlarge,143.01,4.12,recompute,0.05,1.19
3,1024,3B,g6.xlarge,g6.xlarge,112,378.34,0.29,70.00,g6.xlarge->g6.xlarge,308.34,5.40,recompute,0.11,1.56
4,2048,3B,g6.xlarge,g6.xlarge,224,757.22,0.29,121.70,g6.xlarge->g6.xlarge,635.52,6.22,recompute,0.22,1.80


## 1. 전체 비교 테이블
config(model, pair) × context 별 migration/recompute 시간, 차이, 비율, 승자.

In [2]:
view = df[['model', 'pair', 'ctx', 'kv_MiB', 'migrate_ms', 'recompute_ms',
           'diff_ms', 'ratio', 'faster']]
with pd.option_context('display.max_rows', None):
    display(view)

,model,pair,ctx,kv_MiB,migrate_ms,recompute_ms,diff_ms,ratio,faster
0,3B,g6.xlarge->g6.xlarge,128,14,44.98,37.80,7.18,1.19,recompute
1,3B,g6.xlarge->g6.xlarge,256,28,94.23,38.00,56.23,2.48,recompute
2,3B,g6.xlarge->g6.xlarge,512,56,188.91,45.90,143.01,4.12,recompute
3,3B,g6.xlarge->g6.xlarge,1024,112,378.34,70.00,308.34,5.40,recompute
4,3B,g6.xlarge->g6.xlarge,2048,224,757.22,121.70,635.52,6.22,recompute
5,3B,g6.xlarge->g6.xlarge,4096,448,"1,513.77",253.20,"1,260.57",5.98,recompute
6,3B,g6.xlarge->g6.xlarge,8192,896,"3,028.26",574.60,"2,453.66",5.27,recompute
7,3B,g6.xlarge->g6.xlarge,16384,1792,"6,057.26","2,312.20","3,745.06",2.62,recompute
8,3B,g6.xlarge->g6.xlarge,32768,3584,"12,122.50","6,382.20","5,740.30",1.90,recompute
9,3B,g6.xlarge->g6.xlarge,65536,7168,"24,231.85","20,002.80","4,229.05",1.21,recompute


## 2. config 별 요약
각 (model, pair) 에서 migration 이 이긴 횟수, ratio 통계, migration 최선/최악 지점.

In [3]:
rows = []
for (m, p), g in df.groupby(['model', 'pair']):
    gv = g.dropna(subset=['recompute_ms'])
    n_oom = int(g['recompute_ms'].isna().sum())
    mig_wins = int((gv['migrate_ms'] < gv['recompute_ms']).sum())
    best = gv.loc[gv['ratio'].idxmin()] if len(gv) else None      # migration 에 가장 유리
    worst = gv.loc[gv['ratio'].idxmax()] if len(gv) else None     # migration 에 가장 불리
    rows.append({
        'model': m, 'pair': p,
        'n_ctx': len(g), 'mig_wins': mig_wins, 'recomp_wins': len(gv) - mig_wins,
        'oom(=mig only)': n_oom,
        'ratio_min': gv['ratio'].min(), 'ratio_median': gv['ratio'].median(), 'ratio_max': gv['ratio'].max(),
        'best_ctx(min ratio)': int(best['ctx']) if best is not None else None,
        'worst_ctx(max ratio)': int(worst['ctx']) if worst is not None else None,
    })
summary = pd.DataFrame(rows)
display(summary)

,model,pair,n_ctx,mig_wins,recomp_wins,oom(=mig only),ratio_min,ratio_median,ratio_max,best_ctx(min ratio),worst_ctx(max ratio)
0,3B,g6.xlarge->g6.xlarge,10,0,10,0,1.19,3.37,6.22,128,2048
1,3B,g6.xlarge->g6e.xlarge,10,0,10,0,2.27,11.71,18.05,128,4096
2,3B,g6e.xlarge->g6.xlarge,10,0,10,0,1.19,3.36,6.22,128,2048
3,3B,g6e.xlarge->g6e.xlarge,10,0,10,0,2.43,11.65,18.06,128,4096
4,70B,g6.xlarge->g6.xlarge,10,9,0,1,0.02,0.61,0.76,128,8192
5,70B,g6e.xlarge->g6e.xlarge,10,3,7,0,0.29,1.97,2.81,128,4096
6,8B,g6.xlarge->g6.xlarge,10,1,7,2,0.66,1.74,3.16,128,2048
7,8B,g6.xlarge->g6e.xlarge,10,0,10,0,1.60,6.43,10.68,128,4096
8,8B,g6e.xlarge->g6.xlarge,10,1,7,2,0.67,1.74,3.16,128,2048
9,8B,g6e.xlarge->g6e.xlarge,10,0,10,0,1.63,6.44,10.68,128,4096


## 3. Crossover 분석
context 가 커질수록 recompute 는 superlinear, migration 은 linear 라 승자가 바뀐다.
각 config 에서 `migrate_ms - recompute_ms` 의 부호가 바뀌는 구간을 찾고, log-log 보간으로 crossover context 를 추정한다.

In [4]:
def crossovers(g):
    """부호 변화 구간 + log-log 보간 crossover ctx 추정"""
    g = g.dropna(subset=['recompute_ms']).sort_values('ctx')
    recs = g.to_dict('records')
    out = []
    for a, b in zip(recs, recs[1:]):
        da, db = a['diff_ms'], b['diff_ms']
        if (da < 0) != (db < 0):  # 승자 바뀜
            # log(ratio) 가 log(ctx) 에 선형이라 가정, ratio=1 (log=0) 지점
            la, lb = np.log(a['ratio']), np.log(b['ratio'])
            lca, lcb = np.log(a['ctx']), np.log(b['ctx'])
            lc = lca + (0 - la) * (lcb - lca) / (lb - la)
            out.append({
                'from_ctx': a['ctx'], 'to_ctx': b['ctx'],
                'from_winner': a['faster'], 'to_winner': b['faster'],
                'crossover_ctx~': round(float(np.exp(lc))),
            })
    return out

cross_rows = []
for (m, p), g in df.groupby(['model', 'pair']):
    cs = crossovers(g)
    if not cs:
        gv = g.dropna(subset=['recompute_ms'])
        always = gv['faster'].iloc[0] if len(gv) else 'n/a'
        cross_rows.append({'model': m, 'pair': p, 'note': f'crossover 없음 (항상 {always} 우세)',
                           'from_ctx': None, 'to_ctx': None, 'crossover_ctx~': None})
    else:
        for c in cs:
            cross_rows.append({'model': m, 'pair': p,
                               'note': f"{c['from_winner']} -> {c['to_winner']}",
                               'from_ctx': c['from_ctx'], 'to_ctx': c['to_ctx'],
                               'crossover_ctx~': c['crossover_ctx~']})
display(pd.DataFrame(cross_rows))

,model,pair,note,from_ctx,to_ctx,crossover_ctx~
0,3B,g6.xlarge->g6.xlarge,crossover 없음 (항상 recompute 우세),NaN,NaN,NaN
1,3B,g6.xlarge->g6e.xlarge,crossover 없음 (항상 recompute 우세),NaN,NaN,NaN
2,3B,g6e.xlarge->g6.xlarge,crossover 없음 (항상 recompute 우세),NaN,NaN,NaN
3,3B,g6e.xlarge->g6e.xlarge,crossover 없음 (항상 recompute 우세),NaN,NaN,NaN
4,70B,g6.xlarge->g6.xlarge,crossover 없음 (항상 migration 우세),NaN,NaN,NaN
5,70B,g6e.xlarge->g6e.xlarge,migration -> recompute,256.00,512.00,332.00
6,70B,g6e.xlarge->g6e.xlarge,recompute -> migration,"32,768.00","65,536.00","58,264.00"
7,8B,g6.xlarge->g6.xlarge,migration -> recompute,128.00,256.00,190.00
8,8B,g6.xlarge->g6e.xlarge,crossover 없음 (항상 recompute 우세),NaN,NaN,NaN
9,8B,g6e.xlarge->g6.xlarge,migration -> recompute,128.00,256.00,189.00


## 4. Recompute 불가(OOM) → migration 필수 케이스
destination instance 에서 그 context 를 recompute 할 수 없는 경우. 시간과 무관하게 migration 이 유일한 옵션.

In [5]:
oom = df[df['recompute_ms'].isna()][['model', 'pair', 'ctx', 'kv_MiB', 'migrate_ms']]
print(f'OOM(=migration 필수) 케이스: {len(oom)}건')
display(oom)

OOM(=migration 필수) 케이스: 5건


,model,pair,ctx,kv_MiB,migrate_ms
49,70B,g6.xlarge->g6.xlarge,65536,512,"1,730.47"
68,8B,g6.xlarge->g6.xlarge,32768,4096,"13,856.79"
69,8B,g6.xlarge->g6.xlarge,65536,8192,"27,695.37"
88,8B,g6e.xlarge->g6.xlarge,32768,4096,"13,861.02"
89,8B,g6e.xlarge->g6.xlarge,65536,8192,"27,695.86"


## 5. Break-even 네트워크 대역폭
현재 측정 대역폭은 거의 균일하게 ~0.29 GB/s. 각 지점에서 **migration 이 recompute 와 같아지려면 필요한 대역폭** `= KV_GB / recompute_s`.
이 값이 실제 대역폭보다 크면(=`required/actual = ratio > 1`) 그만큼 네트워크가 빨라져야 migration 이 유리해진다.

In [6]:
be = df.dropna(subset=['recompute_ms']).copy()
be['required/actual(x)'] = be['breakeven_bw_GBps'] / be['bw']
view_be = be[['model', 'pair', 'ctx', 'kv_GB', 'recompute_ms',
              'bw', 'breakeven_bw_GBps', 'required/actual(x)']]
with pd.option_context('display.max_rows', None):
    display(view_be)
print(f"실제 대역폭 평균: {df['bw'].mean():.3f} GB/s")
print(f"migration 이 유리해지려면 필요한 대역폭 중앙값: {be['breakeven_bw_GBps'].median():.2f} GB/s "
      f"(실제 대비 약 {be['required/actual(x)'].median():.1f}x)")

,model,pair,ctx,kv_GB,recompute_ms,bw,breakeven_bw_GBps,required/actual(x)
0,3B,g6.xlarge->g6.xlarge,128,0.01,37.80,0.30,0.36,1.21
1,3B,g6.xlarge->g6.xlarge,256,0.03,38.00,0.29,0.72,2.48
2,3B,g6.xlarge->g6.xlarge,512,0.05,45.90,0.29,1.19,4.11
3,3B,g6.xlarge->g6.xlarge,1024,0.11,70.00,0.29,1.56,5.39
4,3B,g6.xlarge->g6.xlarge,2048,0.22,121.70,0.29,1.80,6.20
5,3B,g6.xlarge->g6.xlarge,4096,0.44,253.20,0.29,1.73,5.96
6,3B,g6.xlarge->g6.xlarge,8192,0.88,574.60,0.29,1.52,5.25
7,3B,g6.xlarge->g6.xlarge,16384,1.75,"2,312.20",0.29,0.76,2.61
8,3B,g6.xlarge->g6.xlarge,32768,3.50,"6,382.20",0.29,0.55,1.89
9,3B,g6.xlarge->g6.xlarge,65536,7.00,"20,002.80",0.29,0.35,1.21


실제 대역폭 평균: 0.306 GB/s
migration 이 유리해지려면 필요한 대역폭 중앙값: 0.83 GB/s (실제 대비 약 2.9x)


## 6. (참고) 70B per-layer 정규화 view
`plot_comparison.py` 와 동일하게 g6.xlarge=÷2, g6e.xlarge=÷8 로 per-layer 환산. (위 직접 비교의 ratio 는 정규화와 무관하게 동일함을 확인용.)

In [7]:
LAYERS = {'g6.xlarge': 2, 'g6e.xlarge': 8}
g70 = df[df['model'] == '70B'].copy()
g70['nlayer'] = g70['dst'].map(LAYERS)
g70['migrate_ms/layer']   = g70['migrate_ms'] / g70['nlayer']
g70['recompute_ms/layer'] = g70['recompute_ms'] / g70['nlayer']
display(g70[['pair', 'ctx', 'nlayer', 'migrate_ms/layer', 'recompute_ms/layer', 'ratio', 'faster']])

,pair,ctx,nlayer,migrate_ms/layer,recompute_ms/layer,ratio,faster
40,g6.xlarge->g6.xlarge,128,2,0.32,15.30,0.02,migration
41,g6.xlarge->g6.xlarge,256,2,2.22,15.15,0.15,migration
42,g6.xlarge->g6.xlarge,512,2,5.57,16.65,0.33,migration
43,g6.xlarge->g6.xlarge,1024,2,13.48,22.70,0.59,migration
44,g6.xlarge->g6.xlarge,2048,2,27.02,37.75,0.72,migration
45,g6.xlarge->g6.xlarge,4096,2,53.56,71.55,0.75,migration
46,g6.xlarge->g6.xlarge,8192,2,108.17,141.70,0.76,migration
47,g6.xlarge->g6.xlarge,16384,2,216.41,301.00,0.72,migration
48,g6.xlarge->g6.xlarge,32768,2,431.90,707.80,0.61,migration
49,g6.xlarge->g6.xlarge,65536,2,865.24,NaN,NaN,migration (recompute OOM)


## 7. 종합 요약 (수치)

In [8]:
gv = df.dropna(subset=['recompute_ms'])
print('=== 전체 ===')
print(f"비교 가능한 지점: {len(gv)}, 그 중 migration 이 더 빠른 경우: {(gv['migrate_ms']<gv['recompute_ms']).sum()} "
      f"({(gv['migrate_ms']<gv['recompute_ms']).mean()*100:.0f}%)")
print(f"OOM 으로 migration 필수: {df['recompute_ms'].isna().sum()} 건")
print(f"ratio(=migrate/recompute) 중앙값: {gv['ratio'].median():.2f}  (>1 = recompute 우세)")
print()
print('=== model 별 ratio 중앙값 ===')
print(gv.groupby('model')['ratio'].median().round(2).to_string())
print()
print('=== destination instance 별 ratio 중앙값 ===')
print(gv.groupby('dst')['ratio'].median().round(2).to_string())
print()
print('=== migration 이 이긴 지점들 ===')
win = gv[gv['migrate_ms'] < gv['recompute_ms']][['model', 'pair', 'ctx', 'migrate_ms', 'recompute_ms', 'ratio']]
print(win.to_string(index=False) if len(win) else '  (없음)')

=== 전체 ===
비교 가능한 지점: 95, 그 중 migration 이 더 빠른 경우: 14 (15%)
OOM 으로 migration 필수: 5 건
ratio(=migrate/recompute) 중앙값: 2.86  (>1 = recompute 우세)

=== model 별 ratio 중앙값 ===
model
3B    5.32
70B   0.76
8B    3.01

=== destination instance 별 ratio 중앙값 ===
dst
g6.xlarge    1.90
g6e.xlarge   5.87

=== migration 이 이긴 지점들 ===
model                   pair   ctx  migrate_ms  recompute_ms  ratio
  70B   g6.xlarge->g6.xlarge   128        0.64         30.60   0.02
  70B   g6.xlarge->g6.xlarge   256        4.44         30.30   0.15
  70B   g6.xlarge->g6.xlarge   512       11.14         33.30   0.33
  70B   g6.xlarge->g6.xlarge  1024       26.96         45.40   0.59
  70B   g6.xlarge->g6.xlarge  2048       54.04         75.50   0.72
  70B   g6.xlarge->g6.xlarge  4096      107.12        143.10   0.75
  70B   g6.xlarge->g6.xlarge  8192      216.35        283.40   0.76
  70B   g6.xlarge->g6.xlarge 16384      432.82        602.00   0.72
  70B   g6.xlarge->g6.xlarge 32768      863.81      1,415.60   0.61
  